<a href="https://colab.research.google.com/github/salamlakhan7/sprint-03-deep-learning-nlp-transformers/blob/main/NLP/nlp_sentiment_analysis_bow_to_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 01: Business Problem**
 our business problem is to classify the sentiments of IMDB reviews.

# **Step 02: Collecting the Data**

We will be getting our data from Kaggle, the dataset I am using is in CSV format and has already been split into train, test, and validation. You can use the following dataset split the data using train test split.

 Here is the link to the dataset:
https://www.kaggle.com/datasets/columbine/imdb-dataset-sentiment-analysis-in-csv-format

# **Step 3 - Imports & Dataset Loading**

**Concept:** Before that preprocess_text() function can run, two things need to happen first:

 (1) import the libraries it depends on, and

 (2) get your actual CSV

In [2]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# **step 04 - Loading the Article's Kaggle Dataset**

Since the article links a specific Kaggle dataset (already split into train/test/validation CSVs), the standard way to pull it into Colab is via Kaggle's API:

In [3]:
import os

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_8fff57c309c00605c94502a25e9cff4e'

!pip install kaggle --quiet
!kaggle datasets download -d columbine/imdb-dataset-sentiment-analysis-in-csv-format
!unzip imdb-dataset-sentiment-analysis-in-csv-format.zip

Dataset URL: https://www.kaggle.com/datasets/columbine/imdb-dataset-sentiment-analysis-in-csv-format
License(s): world-bank
100% 25.7M/25.7M [00:00<00:00, 70.0MB/s]

Archive:  imdb-dataset-sentiment-analysis-in-csv-format.zip
  inflating: Test.csv                
  inflating: Train.csv               
  inflating: Valid.csv               


In [4]:
import pandas as pd

train_df = pd.read_csv('Train.csv')
test_df = pd.read_csv('Test.csv')
valid_df = pd.read_csv('Valid.csv')

print(train_df.shape, test_df.shape, valid_df.shape)
print(train_df.head())

(40000, 2) (5000, 2) (5000, 2)
                                                text  label
0  I grew up (b. 1965) watching and loving the Th...      0
1  When I put this movie in my DVD player, and sa...      0
2  Why do people who do not know what a particula...      0
3  Even though I have great interest in Biblical ...      0
4  Im a die hard Dads Army fan and nothing will e...      1


# **Step 05 : Text Pre-processing**

**Concept:** Raw review text - as seen in train_df.head() above - contains HTML leftovers, punctuation, capitalization inconsistencies, and filler words ("the," "and," "is") that add noise without adding meaning. This function cleans all of that before any vectorization happens.



**Concept overview:** Text pre-processing = cleaning raw text before any modeling happens, so the model sees consistent, meaningful signal instead of noise. The article breaks it into 5 sub-steps — here's each one, mapped directly to the code you already ran.



## **5.1 - Special Characters Removal**

**What it does:** Strips punctuation, symbols, and non-alphanumeric characters, since these add noise without carrying sentiment meaning.

**Code line:** `text = re.sub('[^a-zA-Z]', ' ', text).lower()` (the `[^a-zA-Z]` part)

**Real-life example:** Think of proofreading a handwritten note before typing it into a spreadsheet - you wouldn't type in the doodles, underlines, or coffee stains. Special character removal is stripping out everything that isn't actual word content.

---

## **5.2 - Lowercasing**

**What it does:** Converts all text to lowercase so "Great," "GREAT," and "great" are treated as the **same word**, not three different ones.

**Code line:** `.lower()` - same line as above, chained on.

**Real-life example:** Like sorting library books alphabetically without caring whether the title was printed in bold caps or regular case on the spine - the *content* matters, not the styling.

**Why this matters mathematically:** without lowercasing, your vocabulary (and thus BoW/TF-IDF column count) would **triple** for common words with mixed-case variants, artificially inflating dimensionality for zero benefit.



## **5.3 - Tokenization**

**What it does:** Splits cleaned text into individual word units.

**Code line:** `words = word_tokenize(text)`

**Real-life example:** Like cutting a sentence into individual Scrabble tiles - each word becomes its own separate piece you can now analyze, count, or rearrange independently.



## **5.4 - Stop Words Removal**

**What it does:** Removes common, low-information words ("the," "a," "an," "is") that appear everywhere and don't help distinguish meaning.

**Code lines:**
```python
stop_words = set(stopwords.words('english'))
words = [word for word in words if word not in stop_words]
```

**Real-life example:** If you were summarizing a book by its most *distinctive* words, you wouldn't list "the," "and," "was" - everyone's book has those. You'd list "detective," "murder," "alibi" - the words that actually tell you what the book is about.

**Direct tie to our own TF-IDF math from earlier:** this is doing *manually, before vectorization* what TF-IDF does *automatically, during* vectorization (down-weighting common words). Doing both isn't redundant - stop word removal shrinks the vocabulary size outright, while TF-IDF further fine-tunes weight among what's left.

---

## **5.5 - Stemming vs. Lemmatization (This Article Chose Lemmatization)**

**What it does:** Reduces words to a base form, so "running," "runs," "ran" all collapse into one shared representation.

**Code lines:**
```python
lemmatizer = WordNetLemmatizer()
words = [lemmatizer.lemmatize(word) for word in words]
```

**The key distinction, worth understanding precisely:**

| | Stemming | Lemmatization |
|---|---|---|
| Method | Chops off suffixes mechanically | Maps to real dictionary root, using part-of-speech |
| Speed | Faster | Slower |
| Accuracy | Can produce non-words (e.g. "studies" → "studi") | Produces real words (e.g. "studies" → "study") |
| Example | "better" → "better" (no change, purely suffix-based) | "better" → "good" (understands it's the comparative of "good") |

**Real-life example:** Stemming is like a librarian who just chops "-ing," "-ed," "-s" off every word title mechanically, without checking if the result is even a real word. Lemmatization is like a librarian who actually looks up each word's dictionary root — slower, but always produces something real and correctly grouped.

**Why the article picked lemmatization:** more accurate results, at the cost of speed — a reasonable tradeoff for a one-time preprocessing pass (as opposed to something needing to run in real-time at scale).





In [5]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

def preprocess_text(text):
    # Remove HTML tags
    text = re.sub('<[^>]*>', '', text)

    # Remove non-alphabetic characters and convert to lowercase
    text = re.sub('[^a-zA-Z]', ' ', text).lower()

    # Tokenize the text
    words = word_tokenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]

    # Combine words back into a single string
    preprocessed_text = ' '.join(words)

    return preprocessed_text

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [8]:
import joblib
import os
import pandas as pd # Added for train_df and test_df manipulation
import re # Added for preprocess_text
import nltk # Added for preprocess_text
from nltk.tokenize import word_tokenize # Added for preprocess_text
from nltk.corpus import stopwords # Added for preprocess_text
from nltk.stem import WordNetLemmatizer # Added for preprocess_text

save_dir = '/content/drive/MyDrive/sprint03_nlp'
os.makedirs(save_dir, exist_ok=True)

split_path = f'{save_dir}/train_test_split.pkl'

if os.path.exists(split_path):
    # Load the already split data and labels
    X_train, X_test, y_train_loaded, y_test_loaded = joblib.load(split_path)
    # Assign to global variables if different names were used in previous steps
    # Note: y_train and y_test are already globally defined, assuming consistency
    print('Loaded train/test split')
else:
    # Define preprocess_text function and its dependencies if split is not found
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)

    def preprocess_text(text):
        # Remove HTML tags
        text = re.sub('<[^>]*>', '', text)

        # Remove non-alphabetic characters and convert to lowercase
        text = re.sub('[^a-zA-Z]', ' ', text).lower()

        # Tokenize the text
        words = word_tokenize(text)

        # Remove stopwords
        stop_words = set(stopwords.words('english'))
        words = [word for word in words if word not in stop_words]

        # Lemmatize the words
        lemmatizer = WordNetLemmatizer()
        words = [lemmatizer.lemmatize(word) for word in words]

        # Combine words back into a single string
        preprocessed_text = ' '.join(words)

        return preprocessed_text

    # Preprocess the text data
    preprocessed_train_text = train_df['text'].apply(preprocess_text)
    preprocessed_test_text = test_df['text'].apply(preprocess_text)

    # Extract labels
    y_train = train_df['label']
    y_test = test_df['label']

    # Assign the globally available preprocessed data to X_train and X_test
    X_train = preprocessed_train_text
    X_test = preprocessed_test_text
    # y_train and y_test are already globally available from previous cells

    joblib.dump((X_train, X_test, y_train, y_test), split_path)
    print('Saved train/test split')

Saved train/test split


# **Step 06: Vectorization / Feature Extraction - Overview**

**Concept:** Once text is cleaned (Step 03), it's still just words - machine learning models need **numbers**. Vectorization is the conversion step: text → numeric vectors. The article names 3 methods, all of which you've already covered in depth (with full math) earlier in this conversation.


-------------------------------------------------------

## **6.1 - Bag-of-Words (BoW)**

**What it is:** Creates a dictionary of every unique word in the corpus, counts occurrences per document.

**Already covered, with full math and your own loan-dataset example.** Recap: simple, good starting point, but treats every word as equally important.



## **6.2 - TF-IDF**

**What it is:** Weights each word by frequency in the document **and** rarity across the corpus.

**Already covered, with full formula** ($TF \times IDF$) and the "loan" (0.10 weight) vs "unauthorized" (3.0 weight) worked example from earlier.



## **6.3 - Word2Vec**

**What it is:** Neural network-based, learns dense embeddings from co-occurrence patterns. Article mentions two training architectures:

| Architecture | What it predicts |
|---|---|
| **CBOW** (Continuous Bag-of-Words) | Given surrounding context words → predict the missing target word |
| **Skip-gram** | Given a target word → predict its surrounding context words |

**Already covered conceptually** (king - man + woman ≈ queen), but this is the first time the CBOW/Skip-gram distinction specifically has come up - worth flagging as new, even though Word2Vec itself isn't.



**Worth keeping in mind going forward:** no single method is universally best - worth experimenting and evaluating per task. This directly matches our own Sprint 03 philosophy so far (RNN vs LSTM vs GRU, none being a strict winner in every case).

---



In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Preprocess the text data
preprocessed_train_text = train_df['text'].apply(preprocess_text)
preprocessed_test_text = test_df['text'].apply(preprocess_text)

# Extract labels
y_train = train_df['label']
y_test = test_df['label']

def train_svm_with_representations(train_data, test_data, y_train_labels, representation):
    if representation == 'bow':
        vectorizer = CountVectorizer()
    elif representation == 'tfidf':
        vectorizer = TfidfVectorizer()
    else:
        raise ValueError("Invalid representation. Choose 'bow' or 'tfidf'.")

    X_train_vectorized = vectorizer.fit_transform(train_data)
    X_test_vectorized = vectorizer.transform(test_data)

    clf = SVC()
    clf.fit(X_train_vectorized, y_train_labels)
    y_pred = clf.predict(X_test_vectorized)

    return y_pred

# Bag of Words
y_pred_bow = train_svm_with_representations(preprocessed_train_text, preprocessed_test_text, y_train, 'bow')
accuracy_bow = accuracy_score(y_test, y_pred_bow)
print(f"Accuracy (Bag of Words): {accuracy_bow}")

Accuracy (Bag of Words): 0.8802


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import joblib
import os

save_dir = '/content/drive/MyDrive/sprint03_nlp'
os.makedirs(save_dir, exist_ok=True)

model_path = f'{save_dir}/svm_bow_model.pkl'
vectorizer_path = f'{save_dir}/bow_vectorizer.pkl'
preds_path = f'{save_dir}/y_pred_bow.pkl'

if os.path.exists(model_path) and os.path.exists(vectorizer_path):
    print('Found saved model — loading instead of retraining...')
    clf = joblib.load(model_path)
    vectorizer = joblib.load(vectorizer_path)
    if os.path.exists(preds_path):
        y_pred_bow = joblib.load(preds_path)
    else:
        X_test_vectorized = vectorizer.transform(preprocessed_test_text)
        y_pred_bow = clf.predict(X_test_vectorized)
        joblib.dump(y_pred_bow, preds_path)
    print('Loaded successfully')
else:
    print('No saved model found — training from scratch...')
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.svm import SVC

    vectorizer = CountVectorizer()
    X_train_vectorized = vectorizer.fit_transform(preprocessed_train_text)

    clf = SVC()
    clf.fit(X_train_vectorized, y_train)

    X_test_vectorized = vectorizer.transform(preprocessed_test_text)
    y_pred_bow = clf.predict(X_test_vectorized)

    joblib.dump(clf, model_path)
    joblib.dump(vectorizer, vectorizer_path)
    joblib.dump(y_pred_bow, preds_path)
    print('Trained and saved successfully')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No saved model found — training from scratch...
Trained and saved successfully


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import joblib, os

save_dir = '/content/drive/MyDrive/sprint03_nlp'
os.makedirs(save_dir, exist_ok=True)

model_path = f'{save_dir}/svm_tfidf_model.pkl'
vectorizer_path = f'{save_dir}/tfidf_vectorizer.pkl'
preds_path = f'{save_dir}/y_pred_tfidf.pkl'

def train_svm_with_representations(train_data, test_data, y_train_labels, representation):
    if representation == 'bow':
        vectorizer = CountVectorizer()
    elif representation == 'tfidf':
        vectorizer = TfidfVectorizer()
    else:
        raise ValueError("Invalid representation. Choose 'bow' or 'tfidf'.")

    X_train_vectorized = vectorizer.fit_transform(train_data)
    X_test_vectorized = vectorizer.transform(test_data)

    clf = SVC()
    clf.fit(X_train_vectorized, y_train_labels)
    y_pred = clf.predict(X_test_vectorized)

    return clf, vectorizer, y_pred

if os.path.exists(model_path) and os.path.exists(vectorizer_path) and os.path.exists(preds_path):
    print('Found saved TF-IDF model — loading instead of retraining...')
    clf_tfidf = joblib.load(model_path)
    vectorizer_tfidf = joblib.load(vectorizer_path)
    y_pred_tfidf = joblib.load(preds_path)
    print('Loaded successfully')
else:
    print('No saved TF-IDF model found — training from scratch...')
    clf_tfidf, vectorizer_tfidf, y_pred_tfidf = train_svm_with_representations(preprocessed_train_text, preprocessed_test_text, y_train, 'tfidf')

    joblib.dump(clf_tfidf, model_path)
    joblib.dump(vectorizer_tfidf, vectorizer_path)
    joblib.dump(y_pred_tfidf, preds_path)
    print('Trained and saved successfully')

accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)
print(f"Accuracy (TF-IDF): {accuracy_tfidf}")

No saved TF-IDF model found — training from scratch...
Trained and saved successfully
0.9004


In [ ]:
!pip install gensim --quiet
from gensim.models import Word2Vec
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np
import joblib, os

save_dir = '/content/drive/MyDrive/sprint03_nlp'
os.makedirs(save_dir, exist_ok=True)

model_path = f'{save_dir}/svm_word2vec_model.pkl'
w2v_train_path = f'{save_dir}/word2vec_train_model.pkl'
preds_path = f'{save_dir}/y_pred_word2vec.pkl'

def get_word2vec_embeddings(data):
    tokenized_sentences = [sentence.split() for sentence in data]
    model = Word2Vec(tokenized_sentences, vector_size=100, window=5, min_count=1, workers=4)
    embeddings = np.array([np.mean([model.wv[word] for word in sentence if word in model.wv], axis=0) for sentence in tokenized_sentences])
    return embeddings, model

def train_svm_with_word2vec(train_data, test_data, y_train_labels):
    X_train, w2v_train_model = get_word2vec_embeddings(train_data)
    X_test, _ = get_word2vec_embeddings(test_data)

    clf = SVC()
    clf.fit(X_train, y_train_labels)
    y_pred = clf.predict(X_test)

    return clf, w2v_train_model, y_pred

if os.path.exists(model_path) and os.path.exists(preds_path):
    print('Found saved Word2Vec model — loading instead of retraining...')
    clf_word2vec = joblib.load(model_path)
    y_pred_word2vec = joblib.load(preds_path)
    print('Loaded successfully')
else:
    print('No saved Word2Vec model found — training from scratch...')
    clf_word2vec, w2v_train_model, y_pred_word2vec = train_svm_with_word2vec(preprocessed_train_text, preprocessed_test_text, y_train)

    joblib.dump(clf_word2vec, model_path)
    joblib.dump(w2v_train_model, w2v_train_path)
    joblib.dump(y_pred_word2vec, preds_path)
    print('Trained and saved successfully')

accuracy_word2vec = accuracy_score(y_test, y_pred_word2vec)
print(f"Accuracy (Custom Word2Vec): {accuracy_word2vec}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 72.2 MB/s eta 0:00:00
No saved Word2Vec model found — training from scratch...
Trained and saved successfully
Accuracy (Custom Word2Vec): 0.499


In [ ]:
import gensim.downloader as api
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import joblib, os

save_dir = '/content/drive/MyDrive/sprint03_nlp'
os.makedirs(save_dir, exist_ok=True)

model_path = f'{save_dir}/svm_google_w2v_model.pkl'
preds_path = f'{save_dir}/y_pred_google_word2vec.pkl'

def get_google_word2vec_embeddings(data, model):
    tokenized_sentences = [sentence.split() for sentence in data]
    embeddings = []
    for sentence in tokenized_sentences:
        sentence_embeddings = [model[word] for word in sentence if word in model]
        if sentence_embeddings:
            embeddings.append(np.mean(sentence_embeddings, axis=0))
        else:
            embeddings.append(np.zeros(300))
    return np.array(embeddings)

def train_svm_with_google_word2vec(train_data, test_data, y_train_labels):
    print('Loading Google News word2vec model (this is a ~1.6GB download, only happens once)...')
    google_model = api.load("word2vec-google-news-300")

    X_train = get_google_word2vec_embeddings(train_data, google_model)
    X_test = get_google_word2vec_embeddings(test_data, google_model)

    clf = SVC()
    clf.fit(X_train, y_train_labels)
    y_pred = clf.predict(X_test)

    return clf, y_pred

if os.path.exists(model_path) and os.path.exists(preds_path):
    print('Found saved Google Word2Vec model — loading instead of retraining...')
    clf_google_word2vec = joblib.load(model_path)
    y_pred_google_word2vec = joblib.load(preds_path)
    print('Loaded successfully')
else:
    print('No saved Google Word2Vec model found — training from scratch...')
    clf_google_word2vec, y_pred_google_word2vec = train_svm_with_google_word2vec(X_train, X_test, y_train)

    joblib.dump(clf_google_word2vec, model_path)
    joblib.dump(y_pred_google_word2vec, preds_path)
    print('Trained and saved successfully')

accuracy_google_word2vec = accuracy_score(y_test, y_pred_google_word2vec)
print(f"Accuracy (Google News Word2Vec): {accuracy_google_word2vec}")

No saved Google Word2Vec model found — training from scratch...
Loading Google News word2vec model (this is a ~1.6GB download, only happens once)...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Trained and saved successfully
Accuracy (Google News Word2Vec): 0.8578


In [ ]:
import re, os, joblib
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

save_dir = '/content/drive/MyDrive/sprint03_nlp/part2'
os.makedirs(save_dir, exist_ok=True)
data_path = f'{save_dir}/preprocessed_data.pkl'

if os.path.exists(data_path):
    print('Found saved preprocessed data — loading instead of reprocessing...')
    data = joblib.load(data_path)
    X_train_seq, X_test_seq, X_valid_seq = data['X_train_seq'], data['X_test_seq'], data['X_valid_seq']
    y_train_cat, y_test_cat, y_valid_cat = data['y_train_cat'], data['y_test_cat'], data['y_valid_cat']
    tokenizer = data['tokenizer']
    max_words, max_len = data['max_words'], data['max_len']
    print('Loaded successfully')
else:
    print('No saved preprocessed data found — processing from scratch...')
    nltk.download('punkt')
    nltk.download('punkt_tab')
    nltk.download('stopwords')
    nltk.download('wordnet')

    def preprocess_text(text):
        text = re.sub('<[^>]*>', '', text)
        text = re.sub('[^a-zA-Z]', ' ', text).lower()
        words = word_tokenize(text)
        stop_words = set(stopwords.words('english'))
        words = [word for word in words if word not in stop_words]
        lemmatizer = WordNetLemmatizer()
        words = [lemmatizer.lemmatize(word) for word in words]
        return ' '.join(words)

    preprocessed_train_text = train_df['text'].apply(preprocess_text)
    preprocessed_test_text = test_df['text'].apply(preprocess_text)
    preprocessed_valid_text = valid_df['text'].apply(preprocess_text)

    y_train, y_test, y_valid = train_df['label'], test_df['label'], valid_df['label']

    max_words = 10000
    max_len = 100
    tokenizer = Tokenizer(num_words=max_words, oov_token='<unk>')
    tokenizer.fit_on_texts(list(preprocessed_train_text) + list(preprocessed_test_text) + list(preprocessed_valid_text))

    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(preprocessed_train_text), maxlen=max_len)
    X_test_seq = pad_sequences(tokenizer.texts_to_sequences(preprocessed_test_text), maxlen=max_len)
    X_valid_seq = pad_sequences(tokenizer.texts_to_sequences(preprocessed_valid_text), maxlen=max_len)

    y_train_cat = to_categorical(y_train, num_classes=2)
    y_test_cat = to_categorical(y_test, num_classes=2)
    y_valid_cat = to_categorical(y_valid, num_classes=2)

    joblib.dump({
        'X_train_seq': X_train_seq, 'X_test_seq': X_test_seq, 'X_valid_seq': X_valid_seq,
        'y_train_cat': y_train_cat, 'y_test_cat': y_test_cat, 'y_valid_cat': y_valid_cat,
        'tokenizer': tokenizer, 'max_words': max_words, 'max_len': max_len
    }, data_path)
    print('Preprocessed and saved successfully')

No saved preprocessed data found — processing from scratch...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Preprocessed and saved successfully


# **step 07 - Vanilla RNN - checkpointed version**

Same idea as before: skip model.fit() (the slow part) if a saved model already exists.

In [ ]:
import os
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

save_dir = '/content/drive/MyDrive/sprint03_nlp/part2'
model_path = f'{save_dir}/vanilla_rnn_model.keras'

if os.path.exists(model_path):
    print('Found saved Vanilla RNN model — loading instead of retraining...')
    model = load_model(model_path)
    print('Loaded successfully')
else:
    print('No saved Vanilla RNN model found — training from scratch...')
    model = Sequential()
    model.add(Embedding(max_words, 128, input_length=max_len))
    model.add(SimpleRNN(64))
    model.add(Dropout(0.5))
    model.add(Dense(2, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    history = model.fit(X_train_seq, y_train_cat, validation_data=(X_valid_seq, y_valid_cat),
                        epochs=25, batch_size=128, callbacks=[early_stop])

    model.save(model_path)
    print('Trained and saved successfully')

loss, accuracy = model.evaluate(X_test_seq, y_test_cat)
print("Test accuracy:", accuracy)

Found saved Vanilla RNN model — loading instead of retraining...
Loaded successfully
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8620 - loss: 0.3336
Test accuracy: 0.8619999885559082


# **Step 08 - LSTM Model - Sentiment Classification**

This step swaps the SimpleRNN layer for an LSTM layer, which handles long-range dependencies in text much better than vanilla RNN thanks to its gating mechanism. It reuses the same preprocessed data (X_train_seq, X_valid_seq, X_test_seq, max_words, max_len) already saved from the setup cell - no re-preprocessing needed. Same checkpoint pattern: skip training if a saved LSTM model already exists.

In [ ]:
import os
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

save_dir = '/content/drive/MyDrive/sprint03_nlp/part2'
model_path = f'{save_dir}/lstm_model.keras'

if os.path.exists(model_path):
    print('Found saved LSTM model — loading instead of retraining...')
    model = load_model(model_path)
    print('Loaded successfully')
else:
    print('No saved LSTM model found — training from scratch...')
    model = Sequential()
    model.add(Embedding(max_words, 128, input_length=max_len))
    model.add(LSTM(64))
    model.add(Dropout(0.5))
    model.add(Dense(2, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model.fit(X_train_seq, y_train_cat, validation_data=(X_valid_seq, y_valid_cat),
                        epochs=10, batch_size=128, callbacks=[early_stop])

    model.save(model_path)
    print('Trained and saved successfully')

loss, accuracy = model.evaluate(X_test_seq, y_test_cat)
print("Test accuracy:", accuracy)

No saved LSTM model found — training from scratch...
Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


313/313 ━━━━━━━━━━━━━━━━━━━━ 68s 210ms/step - accuracy: 0.8387 - loss: 0.3621 - val_accuracy: 0.8842 - val_loss: 0.2869
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 210ms/step - accuracy: 0.9104 - loss: 0.2302 - val_accuracy: 0.8814 - val_loss: 0.3264
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 66s 211ms/step - accuracy: 0.9312 - loss: 0.1806 - val_accuracy: 0.8674 - val_loss: 0.3842
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 66s 212ms/step - accuracy: 0.9479 - loss: 0.1388 - val_accuracy: 0.8640 - val_loss: 0.4246
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 66s 211ms/step - accuracy: 0.9585 - loss: 0.1118 - val_accuracy: 0.8636 - val_loss: 0.4550
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 66s 210ms/step - accuracy: 0.9715 - loss: 0.0814 - val_accuracy: 0.8582 - val_loss: 0.5105
Trained and saved successfully
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.8820 - loss: 0.2818
Test accuracy: 0.8820000290870667


# **Step 09 -GRU Model - Sentiment Classification**

GRU is a lighter alternative to LSTM - it uses just two gates (reset and update) instead of LSTM's three, meaning fewer parameters and generally faster training, at some potential cost to performance on more complex patterns. Same setup as before: reuses the already-saved preprocessed data (X_train_seq, X_valid_seq, X_test_seq, max_words, max_len), and checkpoints the trained model so it never needs retraining once saved.

In [ ]:
import os
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import GRU, Dense, Embedding, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

save_dir = '/content/drive/MyDrive/sprint03_nlp/part2'
model_path = f'{save_dir}/gru_model.keras'

if os.path.exists(model_path):
    print('Found saved GRU model — loading instead of retraining...')
    model = load_model(model_path)
    print('Loaded successfully')
else:
    print('No saved GRU model found — training from scratch...')
    model = Sequential()
    model.add(Embedding(max_words, 128, input_length=max_len))
    model.add(GRU(64))
    model.add(Dropout(0.5))
    model.add(Dense(2, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model.fit(X_train_seq, y_train_cat, validation_data=(X_valid_seq, y_valid_cat),
                        epochs=25, batch_size=128, callbacks=[early_stop])

    model.save(model_path)
    print('Trained and saved successfully')

loss, accuracy = model.evaluate(X_test_seq, y_test_cat)
print("Test accuracy:", accuracy)

No saved GRU model found — training from scratch...
Epoch 1/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 75s 229ms/step - accuracy: 0.8171 - loss: 0.3887 - val_accuracy: 0.8774 - val_loss: 0.2993
Epoch 2/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 72s 230ms/step - accuracy: 0.9062 - loss: 0.2394 - val_accuracy: 0.8814 - val_loss: 0.2970
Epoch 3/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 74s 238ms/step - accuracy: 0.9285 - loss: 0.1857 - val_accuracy: 0.8742 - val_loss: 0.3386
Epoch 4/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 236ms/step - accuracy: 0.9486 - loss: 0.1406 - val_accuracy: 0.8708 - val_loss: 0.3694
Epoch 5/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 72s 229ms/step - accuracy: 0.9621 - loss: 0.1071 - val_accuracy: 0.8646 - val_loss: 0.4206
Epoch 6/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 73s 233ms/step - accuracy: 0.9714 - loss: 0.0830 - val_accuracy: 0.8650 - val_loss: 0.4967
Epoch 7/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 81s 231ms/step - accuracy: 0.9780 - loss: 0.0645 - val_accuracy: 0.8610 - val_loss: 0.5602
Trained and saved successfully


# **Step 10  Bi-Directional LSTM - Sentiment Classification**

This wraps the LSTM layer in a Bidirectional layer, running two LSTMs over the sequence at once - one reading forward, one reading backward - then combining both outputs. This lets the model use context from both sides of a word, not just what came before it, at the cost of roughly double the computation of a regular LSTM. This is the last model in Part 2. Same setup: reuses your saved preprocessed data, checkpoints the trained model.

In [ ]:
import os
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Embedding, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

save_dir = '/content/drive/MyDrive/sprint03_nlp/part2'
model_path = f'{save_dir}/bilstm_model.keras'

if os.path.exists(model_path):
    print('Found saved Bi-Directional LSTM model — loading instead of retraining...')
    model = load_model(model_path)
    print('Loaded successfully')
else:
    print('No saved Bi-Directional LSTM model found — training from scratch...')
    model = Sequential()
    model.add(Embedding(max_words, 128, input_length=max_len))
    model.add(Bidirectional(LSTM(64)))
    model.add(Dense(2, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model.fit(X_train_seq, y_train_cat, validation_data=(X_valid_seq, y_valid_cat),
                        epochs=10, batch_size=128, callbacks=[early_stop])

    model.save(model_path)
    print('Trained and saved successfully')

loss, accuracy = model.evaluate(X_test_seq, y_test_cat)
print("Test accuracy:", accuracy)

No saved Bi-Directional LSTM model found — training from scratch...
Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 120s 371ms/step - accuracy: 0.8289 - loss: 0.3727 - val_accuracy: 0.8778 - val_loss: 0.2964
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 115s 367ms/step - accuracy: 0.9100 - loss: 0.2248 - val_accuracy: 0.8762 - val_loss: 0.3057
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 115s 367ms/step - accuracy: 0.9379 - loss: 0.1657 - val_accuracy: 0.8712 - val_loss: 0.3400
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 113s 362ms/step - accuracy: 0.9539 - loss: 0.1251 - val_accuracy: 0.8666 - val_loss: 0.4000
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 363ms/step - accuracy: 0.9683 - loss: 0.0879 - val_accuracy: 0.8636 - val_loss: 0.4836
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 117s 373ms/step - accuracy: 0.9830 - loss: 0.0524 - val_accuracy: 0.8610 - val_loss: 0.5857
Trained and saved successfully
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.8846 - loss: 0.2868
Test accuracy: 0.8845999836921692

# **DistilBERT - Pre-trained Transformer for Sentiment Analysis**

This is a different approach from everything so far — instead of training an architecture from scratch, DistilBERT is a pre-trained transformer (already trained on massive text data) that gets fine-tuned on your review data. It should need far less training to reach strong accuracy, but each step is heavier: bigger model, bigger downloads, more compute per batch.

Two real problems in the code as written, worth fixing before running:

TPU requirement. TPUClusterResolver.connect() will hard-fail if your Colab runtime isn't set to TPU (yours has been running on GPU this whole time, based on your step times). I made the strategy detection automatic — it uses TPU if available, otherwise falls back to your current GPU/CPU, so you don't have to touch Colab's runtime settings.
Missing attention_mask. The train/val datasets only pass input_ids, dropping attention_mask entirely — meaning the model can't tell real tokens from padding during training. This was likely a copy-paste slip in the article; I added it back in, since it directly affects accuracy on padded sequences.

I also added the checkpoint pattern (fine-tuning DistilBERT is slow, so this matters even more here) using Hugging Face's own save_pretrained/from_pretrained.

In [9]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.model_selection import train_test_split

save_dir = '/content/drive/MyDrive/sprint03_nlp/part3'
os.makedirs(save_dir, exist_ok=True)
model_dir = f'{save_dir}/distilbert_model'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_len = 512

# Combine existing dataframes and rename columns to match expected format
df = pd.concat([train_df, test_df, valid_df])
df.rename(columns={'text': 'review', 'label': 'sentiment'}, inplace=True)

# Split into train, test and validation sets
X_train, X_test, y_train, y_test = train_test_split(df['review'], df['sentiment'], test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

train_enc = tokenizer(list(X_train), truncation=True, padding=True, max_length=max_len)
valid_enc = tokenizer(list(X_valid), truncation=True, padding=True, max_length=max_len)
test_enc = tokenizer(list(X_test), truncation=True, padding=True, max_length=max_len)

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_enc, y_train)
valid_dataset = ReviewDataset(valid_enc, y_valid)
test_dataset = ReviewDataset(test_enc, y_test)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {'accuracy': (preds == labels).mean()}

if os.path.exists(model_dir):
    print('Found saved DistilBERT model — loading instead of retraining...')
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    print('Loaded successfully')
else:
    print('No saved DistilBERT model found — training from scratch...')
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    training_args = TrainingArguments(
        output_dir='/content/distilbert_checkpoints',
        num_train_epochs=10,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        learning_rate=5e-5,
        logging_steps=50,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()

    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print('Trained and saved successfully')

# Evaluate on test set
model.eval()
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16)
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        labels = batch.pop('labels').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"Test accuracy: {accuracy:.4f}")

Using device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Trained and saved successfully
Test accuracy: 0.9280


# **Testing DistilBERT on Custom Inputs**

In [11]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_dir = '/content/drive/MyDrive/sprint03_nlp/part3/distilbert_model'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
model.eval()

def predict_sentiment(text):
    inputs = tokenizer(text, truncation=True, padding=True, max_length=512, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    confidence = torch.softmax(outputs.logits, dim=1)[0][pred].item()
    label = 'Positive' if pred == 1 else 'Negative'
    return label, confidence

# Try your own examples
samples = [
    "This movie was absolutely fantastic, one of the best I've seen this year!",
    "Complete waste of time, terrible acting and a boring plot.",
    "It was okay, nothing special but not bad either."
]

for text in samples:
    label, conf = predict_sentiment(text)
    print(f"'{text}'\n → {label} ({conf:.2%} confidence)\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

'This movie was absolutely fantastic, one of the best I've seen this year!'
 → Positive (99.51% confidence)

'Complete waste of time, terrible acting and a boring plot.'
 → Negative (99.93% confidence)

'It was okay, nothing special but not bad either.'
 → Negative (91.11% confidence)

